In [1]:
import os
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import numpy as np
from sklearn.cluster import KMeans
import matplotlib.pyplot as plt
from PIL import Image
import ssl
from torchvision import datasets, transforms
from torch.utils.data import TensorDataset, DataLoader
from sklearn.metrics import normalized_mutual_info_score
from scipy.optimize import linear_sum_assignment
import copy
import pandas as pd
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)


Device: cuda


In [2]:
# ------------------------------------------------
# Utility functions
# ------------------------------------------------
ssl._create_default_https_context = ssl._create_unverified_context

def reconstruct_from_txt(txt_path, output_root, source_dataset):
    """
    Rebuild the dataset EXACTLY according to train_mnist.txt / train_usps.txt.
    It creates .jpg files in digital/train_mnist or digital/train_usps.
    """

    with open(txt_path, "r") as f:
        lines = f.readlines()

    # Group MNIST/USPS images by label
    images_by_label = {i: [] for i in range(10)}
    for img, label in source_dataset:
        images_by_label[label].append(img)

    label_counters = {i: 0 for i in range(10)}

    for line in lines:
        path, label_str = line.strip().split()
        label = int(label_str)

        # Remove "./"
        if path.startswith("./"):
            path = path[2:]

        abs_path = os.path.join(output_root, path)
        os.makedirs(os.path.dirname(abs_path), exist_ok=True)

        idx = label_counters[label]
        if idx >= len(images_by_label[label]):
            print(f"[WARN] Not enough samples for label {label}; duplicating last one")
            img = images_by_label[label][-1]
        else:
            img = images_by_label[label][idx]

        label_counters[label] += 1

        img = img.convert("L").resize((32, 32), Image.BILINEAR)
        img.save(abs_path)

    print("[OK] Reconstructed:", txt_path)

def reconstruct_authors_data(root="./datasets/data"):
    """
    Recreate ALL MNIST/USPS images needed for DFC and for DeepFair conversion.
    Must be executed FIRST.
    """

    print("Downloading raw MNIST and USPS...")
    mnist = datasets.MNIST(root, train=True, download=True)
    usps = datasets.USPS(root, train=True, download=True)

    reconstruct_from_txt(
        txt_path=os.path.join(root, "train_mnist.txt"),
        output_root=root,
        source_dataset=mnist
    )

    reconstruct_from_txt(
        txt_path=os.path.join(root, "train_usps.txt"),
        output_root=root,
        source_dataset=usps
    )

    print("\n[ALL DONE] Image reconstruction complete.")
 
def build_fast_mnist_usps(out_dir="./datasets/MNIST_USPS_fast", img_size=28):
    os.makedirs(out_dir, exist_ok=True)

    # Transform: grayscale → resize → flatten
    tf = transforms.Compose([
        transforms.Resize((img_size, img_size)),
        transforms.ToTensor()
    ])

    print("Loading MNIST...")
    mnist = datasets.MNIST("./datasets/data", train=True, download=True, transform=tf)

    print("Loading USPS...")
    usps = datasets.USPS("./datasets/data", train=True, download=True, transform=tf)

    X = []
    y = []
    S = []  # 0 = MNIST, 1 = USPS

    for img, label in mnist:
        X.append(img.numpy().reshape(-1))
        y.append(label)
        S.append(0)

    for img, label in usps:
        X.append(img.numpy().reshape(-1))
        y.append(label)
        S.append(1)

    X = np.array(X, dtype=np.float32)
    y = np.array(y, dtype=np.int64)
    S = np.array(S, dtype=np.int64)

    # Shuffle
    idx = np.random.permutation(len(X))
    X, y, S = X[idx], y[idx], S[idx]

    np.save(os.path.join(out_dir, "X.npy"), X)
    np.save(os.path.join(out_dir, "y.npy"), y)
    np.save(os.path.join(out_dir, "S.npy"), S)

    print("Saved:", X.shape, y.shape, S.shape)

def build_mnist_usps_numpy(
        root_dir="./datasets/data",
        out_dir="./datasets/MNIST_USPS_deepfair",
        img_size=28
    ):
    """
    Convert the reconstructed MNIST–USPS images into numpy arrays for DeepFair.
    Creates:
        X.npy — images as flattened (N, 784)
        y.npy — digit labels (0–9)
        S.npy — sensitive attribute (0=MNIST, 1=USPS)
    """

    os.makedirs(out_dir, exist_ok=True)

    X_list, y_list, S_list = [], [], []

    subsets = [
        ("train_mnist.txt", 0),  # S = 0
        ("train_usps.txt", 1)   # S = 1
    ]

    for txt_file, domain_flag in subsets:
        txt_path = os.path.join(root_dir, txt_file)
        print(f"\nProcessing {txt_path} (domain={domain_flag})")

        with open(txt_path, "r") as f:
            lines = f.readlines()

        for line in lines:
            parts = line.strip().split()
            if len(parts) < 2:
                continue

            rel_path, label_str = parts[0], parts[1]
            label = int(label_str)

            # Remove "./"
            if rel_path.startswith("./"):
                rel_path = rel_path[2:]

            img_path = os.path.join(root_dir, rel_path)

            img = Image.open(img_path).convert("L")
            img = img.resize((img_size, img_size), Image.BILINEAR)
            arr = np.array(img, dtype=np.float32).reshape(-1) / 255.0

            X_list.append(arr)
            y_list.append(label)
            S_list.append(domain_flag)

    # Convert to arrays
    X = np.stack(X_list, axis=0)
    y = np.array(y_list, dtype=np.int64)
    S = np.array(S_list, dtype=np.int64)

    # Shuffle
    rng = np.random.default_rng(12345)
    perm = rng.permutation(len(X))
    X, y, S = X[perm], y[perm], S[perm]

    # Save
    np.save(os.path.join(out_dir, "X.npy"), X)
    np.save(os.path.join(out_dir, "y.npy"), y)
    np.save(os.path.join(out_dir, "S.npy"), S)

    print("\nSaved DeepFair dataset to:", out_dir)
    print("X:", X.shape, "| y:", y.shape, "| S:", S.shape)

In [3]:
# ------------------------------------------------
# Dataset 
# ------------------------------------------------

build_fast_mnist_usps(out_dir="./datasets/MNIST_USPS_deepfair")

subset_dir =  "./datasets/MNIST_USPS_deepfair"
X = np.load(os.path.join(subset_dir, "X.npy")).astype(np.float32)
y = np.load(os.path.join(subset_dir, "y.npy")).astype(np.int64)
S = np.load(os.path.join(subset_dir, "S.npy")).astype(np.int64)

X_t = torch.tensor(X, dtype=torch.float32)
y_t = torch.tensor(y, dtype=torch.long)
S_t = torch.tensor(S, dtype=torch.long)

ds = TensorDataset(X_t, y_t, S_t)
dl = DataLoader(ds, batch_size=256, shuffle=True)

Loading MNIST...
Loading USPS...
Saved: (67291, 784) (67291,) (67291,)


In [4]:
class Autoencoder(nn.Module):
    """Plain AE: encoder -> decoder (no classifier head)."""
    def __init__(self, input_dim: int, latent_dim: int, negative_slope: float = 0.01):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, 500),  nn.LeakyReLU(negative_slope, inplace=True),
            nn.Linear(500, 500),        nn.LeakyReLU(negative_slope, inplace=True),
            nn.Linear(500, 2000),       nn.LeakyReLU(negative_slope, inplace=True),
            nn.Linear(2000, latent_dim),
            #nn.Tanh(),
            nn.BatchNorm1d(latent_dim)
        )
        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, 2000), nn.LeakyReLU(negative_slope, inplace=True),
            nn.Linear(2000, 500),        nn.LeakyReLU(negative_slope, inplace=True),
            nn.Linear(500, 500),         nn.LeakyReLU(negative_slope, inplace=True),
            nn.Linear(500, input_dim),
        )

    def forward(self, x):
        z = self.encoder(x)
        x_hat = self.decoder(z)
        return x_hat, z

def pretrain_autoencoder(dataloader, input_dim, latent_dim, device, epochs=30, lr=1e-3):
    ae = Autoencoder(input_dim, latent_dim).to(device)
    opt = optim.Adam(ae.parameters(), lr=lr)
    loss_fn = nn.MSELoss()
    for ep in range(epochs):
        total = 0
        nobs = 0
        for batch in dataloader:
            if isinstance(batch, (list, tuple)):
                xb = batch[0]
            else:
                xb = batch
            xb = xb.to(device).float()

            opt.zero_grad(set_to_none=True)
            x_hat, _ = ae(xb)
            loss = loss_fn(x_hat, xb)
            loss.backward()
            opt.step()

            bs = xb.size(0)
            total += loss.item() * bs
            nobs  += bs
        print(f"Pretrain epoch {ep+1}/{epochs} | loss={total/nobs:.6f}")
    return ae


In [5]:
@torch.no_grad()
def init_kmeans_centers_from_latent(ae, dataloader, device, k: int):
    """Run KMeans on current latent to initialize centers."""
    ae.eval()
    Z = []
    for batch in dataloader:
        xb = batch[0] if isinstance(batch, (list, tuple)) else batch
        xb = xb.to(device).float()
        _, z = ae(xb)
        Z.append(z.cpu())
    Z = torch.cat(Z, dim=0).numpy()
    km = KMeans(n_clusters=k, n_init=20, random_state=0).fit(Z)
    centers = torch.tensor(km.cluster_centers_, dtype=torch.float32, device=device, requires_grad=True)
    return torch.nn.Parameter(centers)

In [6]:
@torch.no_grad()
def init_supervised_anchors(ae, dataloader, device, k: int):
    """
    Clever Initialization: 
    Calculates the mean latent vector for each ground-truth class.
    """
    ae.eval()
    # Dictionary to store lists of latent vectors per digit
    latent_accumulator = {i: [] for i in range(k)}
    
    for batch in dataloader:
        xb, yb = batch[0], batch[1] # Extract images and digit labels
        xb = xb.to(device).float()
        
        _, z = ae(xb) # Get latent representation
        z = z.cpu()
        
        # Group latent vectors by their ground-truth label
        for i in range(len(yb)):
            label = yb[i].item()
            if label < k:
                latent_accumulator[label].append(z[i])

    # Calculate the mean for each group
    anchor_means = []
    for i in range(k):
        group_tensors = torch.stack(latent_accumulator[i])
        anchor_means.append(group_tensors.mean(dim=0))
    
    # Stack into a (k, latent_dim) tensor
    centers = torch.stack(anchor_means).to(device)
    centers.requires_grad = True
    
    print(f"✔ Supervised Anchors initialized for {k} clusters.")
    return torch.nn.Parameter(centers)

In [7]:
@torch.no_grad()
def torch_kmeans_cost_full_stream(ae, centers, dataloader, device, reduce: str = "mean"):

    ae.eval()
    centers = centers.to(device)

    total, nobs = 0.0, 0
    for batch in dataloader:
        xb = batch[0] if isinstance(batch, (list, tuple)) else batch
        xb = xb.to(device).float()
        _, z = ae(xb)                           # [B, D]
        d2 = torch.cdist(z, centers).pow(2)     # [B, K]
        mins = d2.min(dim=1).values             # [B,]

        if reduce == "sum":
            total += mins.sum().item()
        else:  
            total += mins.sum().item()
            nobs  += mins.numel()

    if reduce == "sum":
        return total
    else:
        loss = total / max(1, nobs)
        return loss


In [8]:
@torch.enable_grad()
def social_fair_dataset(ae, centers, dataloader, device, eps=1e-12):
    """
    Social Fair objective (dataset-level; hard partition):
      Φ_social = max{ Δ(C, U∩A)/|A|,  Δ(C, U∩B)/|B| },
    where Δ(C, U∩A) = sum_j sum_{x∈U_j∩A} ||z - c_j||^2.
    Returns: scalar with grad + (muA, muB) for logging.
    """
    ae.train()
    sumA = torch.zeros((), device=device)
    sumB = torch.zeros((), device=device)
    nA = 0
    nB = 0

    for batch in dataloader:
        xb = batch[0].to(device).float()
        S  = batch[-1].to(device).long()

        x_hat, z = ae(xb)

        # hard assignments: a_i = argmin_j ||z-c_j||^2
        d2 = torch.cdist(z, centers).pow(2)     # (B,K)
        a  = d2.argmin(dim=1)                   # (B,)
        d2_assigned = (z - centers[a]).pow(2).sum(dim=1)  # (B,)

        A = (S == 1)
        B = ~A
        if A.any():
            sumA = sumA + d2_assigned[A].sum()
            nA  += int(A.sum().item())
        if B.any():
            sumB = sumB + d2_assigned[B].sum()
            nB  += int(B.sum().item())

    muA = sumA / (nA + eps) if nA > 0 else torch.zeros((), device=device)
    muB = sumB / (nB + eps) if nB > 0 else torch.zeros((), device=device)
    social = torch.maximum(muA, muB)  # hard max
    return social, muA, muB

@torch.enable_grad()
def separation_fair_dataset(ae, centers, dataloader, device, eps=1e-12):
    """
    Separation Fair objective (dataset-level):
      Φ_sep = min{ E_A[d^2_bisector], E_B[d^2_bisector] },
    where d^2_bisector is squared distance of z to the bisector between its
    nearest and 2nd-nearest centers.
    Returns: scalar with grad + (muA, muB) for logging.
    """
    ae.train()
    sumA = torch.zeros((), device=device)
    sumB = torch.zeros((), device=device)
    nA = 0
    nB = 0

    for batch in dataloader:
        xb = batch[0].to(device).float()
        S  = batch[-1].to(device).long()

        x_hat, z = ae(xb)

        d2 = torch.cdist(z, centers).pow(2)          # (B,K)
        m1_idx = d2.argmin(dim=1)
        d2_mask = d2.scatter(1, m1_idx.unsqueeze(1), float('inf'))
        m2_idx = d2_mask.argmin(dim=1)

        m1 = centers[m1_idx]
        m2 = centers[m2_idx]
        m  = 0.5 * (m1 + m2)
        v  = m2 - m1
        vn = torch.clamp(torch.linalg.norm(v, dim=1, keepdim=True), min=1e-12)
        vhat = v / vn
        s  = ((z - m) * vhat).sum(dim=1)             # signed distance to bisector
        d2_hp = s * s

        A = (S == 1)
        B = ~A
        if A.any():
            sumA = sumA + d2_hp[A].sum()
            nA  += int(A.sum().item())
        if B.any():
            sumB = sumB + d2_hp[B].sum()
            nB  += int(B.sum().item())

    muA = sumA / (nA + eps) if nA > 0 else torch.zeros((), device=device)
    muB = sumB / (nB + eps) if nB > 0 else torch.zeros((), device=device)
    sep = torch.minimum(muA, muB)  # hard min
    return sep, muA, muB


In [9]:
def separation_fair_dataset_batch(z, centers, S):

    d2 = torch.cdist(z, centers).pow(2)

    m1_idx = d2.argmin(dim=1)
    d2_mask = d2.scatter(1, m1_idx.unsqueeze(1), float('inf'))
    m2_idx = d2_mask.argmin(dim=1)

    m1 = centers[m1_idx]
    m2 = centers[m2_idx]

    m = 0.5 * (m1 + m2)

    v = m2 - m1
    vhat = v / torch.clamp(torch.linalg.norm(v, dim=1, keepdim=True), min=1e-12)

    s = ((z - m) * vhat).sum(dim=1)
    d2_hp = s ** 2

    A = (S == 1)
    B = ~A

    muA = d2_hp[A].mean() if A.any() else torch.tensor(0., device=z.device)
    muB = d2_hp[B].mean() if B.any() else torch.tensor(0., device=z.device)

    sep = torch.minimum(muA, muB)

    return sep, muA, muB

def social_fair_batch(z, centers, S, eps=1e-12):

    d2 = torch.cdist(z, centers).pow(2)

    a = d2.argmin(dim=1)
    d2_assigned = (z - centers[a]).pow(2).sum(dim=1)

    A = (S == 1)
    B = ~A

    muA = d2_assigned[A].mean() if A.any() else torch.tensor(0., device=z.device)
    muB = d2_assigned[B].mean() if B.any() else torch.tensor(0., device=z.device)

    social = torch.maximum(muA, muB)

    return social, muA, muB

In [10]:
def compute_weights(lam, mode):

    if mode == "balanced":
        alpha = lam / 2.0
        beta = lam / 2.0

    elif mode == "social_heavy":
        alpha = 0.75 * lam
        beta = 0.25 * lam

    elif mode == "separation_heavy":
        alpha = 0.25 * lam
        beta = 0.75 * lam

    else:
        raise ValueError("Unknown weighting mode")

    return alpha, beta

In [11]:
# ------------------------------------------------
# Main Deep Unifair training loop
# ------------------------------------------------

def train_deepclust_unifair(
    ae, dataloader, device, input_dim, latent_dim, k=3,
    epochs=50, lr=1e-3,
    alpha=0.1,
    beta=1.0,
    lambda_fair=1.0,
    clip_grad=5.0, 
    lambda_soc=None,
    lambda_sep=None
):

    ae = ae.to(device)
    mse = nn.MSELoss()

    print("---------------------------------")
    print("ALPHA:", alpha,
          "BETA:", beta,
          "λ_soc:", lambda_soc,
          "λ_sep:", lambda_sep)
    print("---------------------------------")

    # initialize cluster centers
    centers = init_supervised_anchors(ae, dataloader, device, k)

    opt = optim.AdamW(
        [{"params": ae.parameters(), "lr": lr},
         {"params": [centers], "lr": lr * 0.3}],
        weight_decay=1e-4
    )

    history = {
        "total": [], "rec": [], "cmp": [],
        "social": [], "sep": [],
        "muA_social": [], "muB_social": [],
        "muA_sep": [], "muB_sep": []
    }

    for ep in range(epochs):

        ae.train()

        tot = rec_tot = cmp_tot = 0.0
        nobs = 0

        for batch in dataloader:

            xb = batch[0].to(device).float()
            Sb = batch[2].to(device)

            x_hat, z = ae(xb)

            # reconstruction
            rec_loss = mse(x_hat, xb)

            # compactness
            d2 = torch.cdist(z, centers).pow(2)
            min_d2 = d2.min(dim=1).values
            cmp_loss = min_d2.mean()

            # fairness terms (batch)
            social_batch, _, _ = social_fair_batch(z, centers, Sb)
            sep_batch, _, _ = separation_fair_dataset_batch(z, centers, Sb)

            # UniFair loss
            loss = (
                alpha * rec_loss
                + beta * cmp_loss
                + lambda_soc * social_batch
                - lambda_sep * sep_batch
            )

            opt.zero_grad(set_to_none=True)
            loss.backward()

            if clip_grad is not None:
                torch.nn.utils.clip_grad_norm_(ae.parameters(), clip_grad)

            opt.step()

            bs = xb.size(0)

            nobs += bs
            rec_tot += rec_loss.item() * bs
            cmp_tot += cmp_loss.item() * bs
            tot += loss.item() * bs

        rec_epoch = rec_tot / nobs
        cmp_epoch = cmp_tot / nobs
        total_epoch = tot / nobs

        # ----- evaluate fairness on full dataset -----

        with torch.no_grad():

            social_term, muA_soc, muB_soc = social_fair_dataset(
                ae, centers, dataloader, device
            )

            sep_term, muA_sep, muB_sep = separation_fair_dataset(
                ae, centers, dataloader, device
            )

        history["total"].append(total_epoch)
        history["rec"].append(rec_epoch)
        history["cmp"].append(cmp_epoch)
        history["social"].append(social_term.item())
        history["sep"].append(sep_term.item())
        history["muA_social"].append(muA_soc.item())
        history["muB_social"].append(muB_soc.item())
        history["muA_sep"].append(muA_sep.item())
        history["muB_sep"].append(muB_sep.item())

        print(
            f"[{ep+1:02d}/{epochs}] "
            f"Rec={rec_epoch:.5f} | "
            f"Cmp={cmp_epoch:.5f} | "
            f"Social(max)={social_term.item():.5f} | "
            f"Sep(min)={sep_term.item():.5f} | "
            f"μA_social={muA_soc.item():.5f} "
            f"μB_social={muB_soc.item():.5f} | "
            f"μA_sep={muA_sep.item():.5f} "
            f"μB_sep={muB_sep.item():.5f}"
        )

    return ae, centers, history

In [12]:
def cluster_accuracy(y_true, y_predicted, cluster_number=None):
    if cluster_number is None:
        cluster_number = max(y_predicted.max(), y_true.max()) + 1
    count_matrix = np.zeros((cluster_number, cluster_number), dtype=np.int64)

    for i in range(y_predicted.size):
        count_matrix[y_predicted[i], y_true[i]] += 1

    row_ind, col_ind = linear_sum_assignment(count_matrix.max() - count_matrix)
    reassignment = dict(zip(row_ind, col_ind))
    accuracy = count_matrix[row_ind, col_ind].sum() / y_predicted.size

    return reassignment, accuracy

def entropy(input):
    epsilon = 1e-5
    entropy = -input * torch.log(input + epsilon)
    entropy = torch.sum(entropy, dim=0)
    return entropy

def balance(predicted, size_0, k=10):
    count = torch.zeros((k, 2))

    for i in range(size_0):
        count[predicted[i], 0] += 1
    for i in range(size_0, predicted.shape[0]):
        count[predicted[i], 1] += 1

    count[count == 0] = 1e-5

    balance_0 = torch.min(count[:, 0] / count[:, 1])
    balance_1 = torch.min(count[:, 1] / count[:, 0])

    en_0 = entropy(count[:, 0] / torch.sum(count[:, 0]))
    en_1 = entropy(count[:, 1] / torch.sum(count[:, 1]))

    return min(balance_0, balance_1).numpy(), en_0.numpy(), en_1.numpy()

In [13]:
def run_lambda_seed_grid(
    ae_pretrained,
    dataloader_wb,
    device,
    latent_dim=20,
    k=3,
    epochs=50,
    lr=0.01,
    alpha=1.0,
    beta=1.0,
    lambda_list=(0.0, 0.2, 0.4, 0.6, 0.8, 1.0),
    seeds=range(10),
    save_dir="deepfair_results_unifair",
    collect_decoded_centers=True,
    weight_mode="balanced"
):

    # metric holders
    min_fair_dict = {lam: [] for lam in lambda_list}
    max_fair_dict = {lam: [] for lam in lambda_list}
    seperation_fairness_gaps = {lam: [] for lam in lambda_list}
    social_fairness_gaps     = {lam: [] for lam in lambda_list}
    kmeans_costs  = {lam: [] for lam in lambda_list}
    recon_costs   = {lam: [] for lam in lambda_list}
    compact_costs = {lam: [] for lam in lambda_list}
    accuracy_dict = {lam: [] for lam in lambda_list}
    nmi_dict      = {lam: [] for lam in lambda_list}
    entropy_dict  = {lam: [] for lam in lambda_list}

    # ===== get input dimension =====
    first_batch = next(iter(dataloader_wb))
    X_first = first_batch[0]
    D = X_first.shape[1]    

    # allocate decoded center tensor
    decoded_centers = None
    if collect_decoded_centers:
        L = len(lambda_list); S = len(list(seeds))
        decoded_centers = np.zeros((L, S, k, D), dtype=np.float32)

    pretrained_state = copy.deepcopy(ae_pretrained.state_dict())

    for li, lam in enumerate(lambda_list):
        print(f"\n=== λ = {lam} ===")

        for si, seed in enumerate(seeds):
            print(f"  -> seed {seed}")
            torch.manual_seed(seed)
            np.random.seed(seed)
            if torch.cuda.is_available():
                torch.cuda.manual_seed_all(seed)

            # reset AE
            ae = copy.deepcopy(ae_pretrained).to(device)
            ae.load_state_dict(pretrained_state)
            xb = next(iter(dataloader_wb))[0].to(device)
            '''x_hat, _ = ae(xb)
            print(((x_hat - xb)**2).mean().item())'''
            
            '''for p in ae.encoder.parameters():
                p.requires_grad = False

            for p in ae.decoder.parameters():
                p.requires_grad = False'''

            # train model
            lambda_soc, lambda_sep = compute_weights(lam, mode=weight_mode)
            ae, centers, history = train_deepclust_unifair(
                ae, dataloader_wb, device,
                input_dim=D,
                latent_dim=latent_dim,
                k=k,
                lr=lr,
                epochs=epochs,
                alpha=alpha,
                beta=beta,
                lambda_fair=lam,
                lambda_soc=lambda_soc,
                lambda_sep=lambda_sep
            )
            print("Cluster centers shape:", centers.shape)
            # evaluate metrics
            kmeans = torch_kmeans_cost_full_stream(ae, centers, dataloader_wb, device)
            minFair, cfdA, cfdB = separation_fair_dataset(ae, centers, dataloader_wb, device)
            maxFair, muA, muB   = social_fair_dataset(ae, centers, dataloader_wb, device)

            # ---- Compute predicted cluster assignments ----
            ae.eval()
            Z_all = []
            y_all = []
            S_all = []

            for batch in dataloader_wb:
                xb, yb, Sb = batch
                xb = xb.to(device)

                # encode latent
                _, z = ae(xb)

                #Z_all.append(z.cpu().numpy())
                Z_all.append(z.detach().cpu().numpy())
                y_all.append(yb.numpy())  # true digit
                S_all.append(Sb.numpy())  # domain flag MNIST=0 USPS=1

            Z_all = np.concatenate(Z_all, axis=0)
            y_all = np.concatenate(y_all, axis=0)
            S_all = np.concatenate(S_all, axis=0)

            # distances to each cluster center
            cent = centers.detach().cpu().numpy()
            d2 = ((Z_all[:, None, :] - cent[None, :, :]) ** 2).sum(axis=2)

            pred_clusters = d2.argmin(axis=1)
            _, acc = cluster_accuracy(y_all, pred_clusters, cluster_number=k)
            nmi = normalized_mutual_info_score(y_all, pred_clusters)
            size_0 = (S_all == 0).sum()

            pred_torch = torch.tensor(pred_clusters)
            bal, en0, en1 = balance(pred_torch, size_0, k=k)                                    


            kmeans_costs[lam].append(kmeans)
            min_fair_dict[lam].append(minFair)
            max_fair_dict[lam].append(maxFair)
            seperation_fairness_gaps[lam].append(abs(cfdA - cfdB))
            social_fairness_gaps[lam].append(abs(muA - muB))
            recon_costs[lam].append(history["rec"][-1])
            compact_costs[lam].append(history["cmp"][-1])

            accuracy_dict[lam].append(acc)
            nmi_dict[lam].append(nmi)
            entropy_dict[lam].append((bal, en0, en1))

            # decode cluster centers
            if collect_decoded_centers:
                ae.eval()
                decoded = ae.decoder(centers).detach().cpu().numpy()
                decoded_centers[li, si] = decoded
                
            import os

            os.makedirs(save_dir, exist_ok=True)

            partial_result = {
                "lambda": lam,
                "seed": seed,
                "accuracy": acc,
                "nmi": nmi,
                "min_fair": minFair.item(),
                "max_fair": maxFair.item(),
                "bal": bal,
                "entropy_0": en0,
                "entropy_1": en1,
                "sep_gap": abs(cfdA - cfdB),
                "soc_gap": abs(muA - muB),
                "kmeans": kmeans,
                "rec": history["rec"][-1],
                "cmp": history["cmp"][-1],
            }
            save_dir_mode = os.path.join(save_dir, weight_mode)
            os.makedirs(save_dir_mode, exist_ok=True)
            file_name = f"{save_dir_mode}/result_lambda{lam}_seed{seed}.pt"
            torch.save(partial_result, file_name)

            print("Saved:", file_name)
    results = {
        "min_fair": min_fair_dict,
        "max_fair": max_fair_dict,
        "seperation_fair_gap": seperation_fairness_gaps,
        "social_fair_gap": social_fairness_gaps,
        "kmeans": kmeans_costs,
        "rec": recon_costs,
        "cmp": compact_costs,
        "accuracy": accuracy_dict,
        "nmi": nmi_dict,
        "entropy": entropy_dict
    }
    # SAVE RESULTS
    import os
    os.makedirs(save_dir, exist_ok=True)
    torch.save(results, os.path.join(save_dir_mode, "results.pth"))
    np.save(os.path.join(save_dir_mode, "decoded_centers.npy"), decoded_centers)
    np.save(os.path.join(save_dir_mode, "lambda_vals.npy"), np.array(list(lambda_list)))

    print(f"\n✔ Saved results.pth and decoded centers to: {save_dir}")

   
    
    return results, decoded_centers, list(lambda_list)

In [14]:
# ------------------------------------------------
# Pretraining
# ------------------------------------------------
latent_dim = 20
k = 10
ae = pretrain_autoencoder(dl, 784, latent_dim=20, device=device, epochs=50)

Pretrain epoch 1/50 | loss=0.028953
Pretrain epoch 2/50 | loss=0.016916
Pretrain epoch 3/50 | loss=0.014510
Pretrain epoch 4/50 | loss=0.013301
Pretrain epoch 5/50 | loss=0.012311
Pretrain epoch 6/50 | loss=0.011739
Pretrain epoch 7/50 | loss=0.011210
Pretrain epoch 8/50 | loss=0.010771
Pretrain epoch 9/50 | loss=0.010460
Pretrain epoch 10/50 | loss=0.010217
Pretrain epoch 11/50 | loss=0.009878
Pretrain epoch 12/50 | loss=0.009655
Pretrain epoch 13/50 | loss=0.009407
Pretrain epoch 14/50 | loss=0.009279
Pretrain epoch 15/50 | loss=0.009045
Pretrain epoch 16/50 | loss=0.008873
Pretrain epoch 17/50 | loss=0.008819
Pretrain epoch 18/50 | loss=0.008604
Pretrain epoch 19/50 | loss=0.008505
Pretrain epoch 20/50 | loss=0.008319
Pretrain epoch 21/50 | loss=0.008235
Pretrain epoch 22/50 | loss=0.008187
Pretrain epoch 23/50 | loss=0.008040
Pretrain epoch 24/50 | loss=0.007924
Pretrain epoch 25/50 | loss=0.007833
Pretrain epoch 26/50 | loss=0.007754
Pretrain epoch 27/50 | loss=0.007670
Pretrain e

In [15]:
# ------------------------------------------------
# Example usage
# ------------------------------------------------
results, decoded_centers, lambdas = run_lambda_seed_grid(
    ae,
    dataloader_wb=dl,
    device=device,
    latent_dim=20,
    k=10,
    epochs=50,
    lambda_list=[0.0],#, 0.2, 0.4, 0.6, 0.8, 1.0],
    seeds=[0],
    alpha=1.0, 
    beta=1.0,
    lr=5e-5,
    weight_mode="balanced"
)


=== λ = 0.0 ===
  -> seed 0
---------------------------------
ALPHA: 1.0 BETA: 1.0 λ_soc: 0.0 λ_sep: 0.0
---------------------------------
✔ Supervised Anchors initialized for 10 clusters.
[01/50] Rec=0.00913 | Cmp=3.45311 | Social(max)=2.90206 | Sep(min)=0.76428 | μA_social=2.13734 μB_social=2.90206 | μA_sep=0.95369 μB_sep=0.76428
[02/50] Rec=0.01350 | Cmp=2.49951 | Social(max)=2.27345 | Sep(min)=1.02025 | μA_social=1.82351 μB_social=2.27345 | μA_sep=1.53128 μB_sep=1.02025
[03/50] Rec=0.01627 | Cmp=2.02235 | Social(max)=1.87392 | Sep(min)=1.22425 | μA_social=1.60103 μB_social=1.87392 | μA_sep=1.74545 μB_sep=1.22425
[04/50] Rec=0.01888 | Cmp=1.70992 | Social(max)=1.59945 | Sep(min)=1.30573 | μA_social=1.45547 μB_social=1.59945 | μA_sep=1.84312 μB_sep=1.30573
[05/50] Rec=0.02059 | Cmp=1.47916 | Social(max)=1.39619 | Sep(min)=1.32869 | μA_social=1.24173 μB_social=1.39619 | μA_sep=1.77203 μB_sep=1.32869
[06/50] Rec=0.02204 | Cmp=1.29256 | Social(max)=1.22280 | Sep(min)=1.31726 | μA_socia

In [16]:
save_root = "deepfair_results_unifair"

modes = ["balanced", "social_heavy", "separation_heavy"]

for mode in modes:

    mode_dir = os.path.join(save_root, mode)

    if not os.path.exists(mode_dir):
        continue

    rows = []

    for f in os.listdir(mode_dir):

        if not f.endswith(".pt"):
            continue

        r = torch.load(os.path.join(mode_dir, f), weights_only=False)

        rows.append({
            "lambda": r["lambda"],
            "seed": r["seed"],
            "accuracy": r["accuracy"],
            "nmi": r["nmi"],
            "min_fair": r["min_fair"],
            "max_fair": r["max_fair"],
            "bal": r["bal"],
            "entropy_0": r["entropy_0"],
            "entropy_1": r["entropy_1"],
            "sep_gap": r["sep_gap"],
            "soc_gap": r["soc_gap"],
            "kmeans": r["kmeans"],
            "rec": r["rec"],
            "cmp": r["cmp"],
        })

    results = {"rows": rows}

    torch.save(results, os.path.join(mode_dir, "results.pth"))

    print(f"Saved results.pth for mode: {mode}")

Saved results.pth for mode: balanced
Saved results.pth for mode: social_heavy
Saved results.pth for mode: separation_heavy
